# Practical 5A — CIFAR-10 ResNet50 Transfer Learning

## Aim
Apply ResNet50 feature extraction and fine-tune selected layers for CIFAR-10 classification.

## Setup
Run the installation cell if required. These notebooks use Python 3.10–3.12 where possible. On Apple Silicon, TensorFlow/PyTorch acceleration depends on the installed build; CPU execution is supported.

**Important:** dataset downloads and model training may take time. Results are generated when you run the notebook; no accuracy values are pre-filled.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn tensorflow

## Dependencies and notes\nRequires TensorFlow and an internet connection to download CIFAR-10 and ImageNet pretrained weights. If ImageNet weights cannot be downloaded, change `weights='imagenet'` to `weights=None` but explain that this is no longer pretrained transfer learning.

## 1. Load CIFAR-10 and resize for ResNet50

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
np.random.seed(42); tf.random.set_seed(42)
(x_train,y_train),(x_test,y_test)=keras.datasets.cifar10.load_data()
# Resize to 96x96 for a more useful ResNet input; use subsets to limit memory/time.
x_train=tf.image.resize(x_train[:12000],(96,96)).numpy()
x_test=tf.image.resize(x_test[:3000],(96,96)).numpy()

## 2. Build ResNet50 feature extractor\nImageNet pretrained weights require an internet download on first run. The preprocessing function is required for ResNet50.

In [ ]:
base=keras.applications.ResNet50(include_top=False,weights="imagenet",input_shape=(96,96,3))
base.trainable=False
inp=keras.Input((96,96,3))
x=keras.applications.resnet50.preprocess_input(inp)
x=base(x,training=False)
x=layers.GlobalAveragePooling2D()(x)
x=layers.Dense(256,activation="relu")(x)
out=layers.Dense(10,activation="softmax")(x)
model=keras.Model(inp,out)
model.compile(optimizer="adam",loss="sparse_categorical_crossentropy",metrics=["accuracy"])

## 3. Feature extraction training

In [ ]:
start=time.time()
h1=model.fit(x_train,y_train[:12000],epochs=5,batch_size=64,validation_split=.1,verbose=1)
print("Feature extraction seconds:",round(time.time()-start,1))
print("Feature extraction test:",model.evaluate(x_test,y_test[:3000],verbose=0))

## 4. Fine-tune selected ResNet50 layers\nUnfreeze only the final convolutional stage and use a small learning rate. Batch normalization layers remain frozen for stability.

In [ ]:
base.trainable=True
for layer in base.layers:
    if not layer.name.startswith("conv5_"):
        layer.trainable=False
    if isinstance(layer,layers.BatchNormalization):
        layer.trainable=False
model.compile(optimizer=keras.optimizers.Adam(1e-5),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
start=time.time()
h2=model.fit(x_train,y_train[:12000],epochs=3,batch_size=64,validation_split=.1,verbose=1)
print("Fine-tuning seconds:",round(time.time()-start,1))
print("Fine-tuned test:",model.evaluate(x_test,y_test[:3000],verbose=0))

In [ ]:
plt.plot(h1.history["val_accuracy"],label="Feature extraction")
plt.plot(np.arange(len(h1.history["val_accuracy"]),len(h1.history["val_accuracy"])+len(h2.history["val_accuracy"])),h2.history["val_accuracy"],label="Fine-tuning")
plt.xlabel("Epoch"); plt.ylabel("Validation accuracy"); plt.legend(); plt.show()

## 5. Conclusion\nFeature extraction keeps the pretrained convolutional base frozen; fine-tuning updates selected upper layers with a low learning rate. CIFAR-10 differs from ImageNet, and the resized small subset limits the experiment.